<a href="https://colab.research.google.com/github/emManab/AgencyAi/blob/main/pptgenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall google-generativeai -y
!pip install -U google-genai

In [ ]:
import os
import re
import json
import requests

from google import genai

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE

In [ ]:
import google
from google import genai

print(genai)
print(genai.__file__)

<module 'google.genai' from '/usr/local/lib/python3.12/dist-packages/google/genai/__init__.py'>
/usr/local/lib/python3.12/dist-packages/google/genai/__init__.py


In [ ]:
class ProfessionalPPTGenerator:

    # ========================================================
    # INITIALIZATION
    # ========================================================
    def __init__(self, api_key=None, pexels_api_key=None):

        self.api_key = api_key
        self.pexels_api_key = pexels_api_key

        if not self.api_key:
            raise ValueError("Gemini API key not provided")

        if not self.pexels_api_key:
            raise ValueError("Pexels API key not provided")

        # -------------------------------
        # Gemini
        # -------------------------------
        self.client = genai.Client(
            api_key=self.api_key
        )

        # Use the model available to your project
        self.model = "gemini-3.1-flash-lite"

        # -------------------------------
        # PowerPoint
        # -------------------------------
        self.presentation = Presentation()

        # 16:9
        self.presentation.slide_width = Inches(13.333)
        self.presentation.slide_height = Inches(7.5)

        # -------------------------------
        # Colors
        # -------------------------------
        self.colors = {
            "bg": RGBColor(248, 250, 252),
            "white": RGBColor(255, 255, 255),

            "primary": RGBColor(15, 23, 42),
            "secondary": RGBColor(51, 65, 85),
            "muted": RGBColor(100, 116, 139),

            "accent": RGBColor(37, 99, 235),
            "accent_light": RGBColor(219, 234, 254),

            "border": RGBColor(226, 232, 240),
            "card": RGBColor(255, 255, 255),

            "dark": RGBColor(15, 23, 42),
            "dark_2": RGBColor(30, 41, 59),

            "success": RGBColor(22, 163, 74),
            "warning": RGBColor(234, 88, 12),
        }

        self.title_font = "Aptos Display"
        self.body_font = "Aptos"

    # ========================================================
    # BASIC HELPERS
    # ========================================================
    def _set_background(self, slide, color=None):

        if color is None:
            color = self.colors["bg"]

        fill = slide.background.fill
        fill.solid()
        fill.fore_color.rgb = color

    def _add_text(
        self,
        slide,
        text,
        left,
        top,
        width,
        height,
        font_size=18,
        bold=False,
        color=None,
        alignment=PP_ALIGN.LEFT,
        font_name=None,
        valign=MSO_ANCHOR.TOP
    ):

        if color is None:
            color = self.colors["primary"]

        if font_name is None:
            font_name = self.body_font

        box = slide.shapes.add_textbox(
            left,
            top,
            width,
            height
        )

        tf = box.text_frame
        tf.clear()
        tf.word_wrap = True
        tf.vertical_anchor = valign

        p = tf.paragraphs[0]

        p.text = text
        p.alignment = alignment

        p.font.name = font_name
        p.font.size = Pt(font_size)
        p.font.bold = bold
        p.font.color.rgb = color

        return box

    def _add_card(
        self,
        slide,
        left,
        top,
        width,
        height,
        fill_color=None,
        border_color=None
    ):

        if fill_color is None:
            fill_color = self.colors["card"]

        if border_color is None:
            border_color = self.colors["border"]

        shape = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            left,
            top,
            width,
            height
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = fill_color

        shape.line.color.rgb = border_color
        shape.line.width = Pt(1)

        try:
            shape.adjustments[0] = 0.08
        except Exception:
            pass

        return shape

    def _add_accent(
        self,
        slide,
        left,
        top,
        width=Inches(0.8),
        height=Inches(0.07)
    ):

        shape = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE,
            left,
            top,
            width,
            height
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = self.colors["accent"]

        shape.line.fill.background()

        return shape

    def _add_slide_number(self, slide, number):

        self._add_text(
            slide,
            f"{number:02d}",
            Inches(12.05),
            Inches(6.95),
            Inches(0.65),
            Inches(0.25),
            font_size=10,
            bold=True,
            color=self.colors["muted"],
            alignment=PP_ALIGN.RIGHT
        )

    def _add_footer(
        self,
        slide,
        text="RESIDENTIAL STATUS • INCOME TAX"
    ):

        self._add_text(
            slide,
            text,
            Inches(0.65),
            Inches(6.98),
            Inches(5.5),
            Inches(0.2),
            font_size=8,
            bold=True,
            color=self.colors["muted"]
        )

    # ========================================================
    # FORMATTED TEXT
    # ========================================================
    def _add_formatted_text(
        self,
        paragraph,
        text,
        base_size=18,
        dark=False
    ):

        parts = re.split(
            r"\*\*(.*?)\*\*",
            str(text)
        )

        for index, part in enumerate(parts):

            if not part:
                continue

            run = paragraph.add_run()
            run.text = part

            run.font.name = self.body_font
            run.font.size = Pt(base_size)

            if dark:

                if index % 2 == 1:
                    run.font.bold = True
                    run.font.color.rgb = self.colors["accent"]
                else:
                    run.font.color.rgb = RGBColor(
                        226, 232, 240
                    )

            else:

                if index % 2 == 1:
                    run.font.bold = True
                    run.font.color.rgb = self.colors["accent"]
                else:
                    run.font.color.rgb = self.colors["secondary"]

    # ========================================================
    # GENERATE OUTLINE
    # ========================================================
    def generate_content_outline(
        self,
        topic,
        num_slides=5
    ):

        prompt = f"""
Create a polished, visually engaging PowerPoint presentation
about:

"{topic}"

Create exactly {num_slides} slides.

Return ONLY valid JSON.

Use this exact structure:

[
  {{
    "title": "Slide title",
    "subtitle": "Short subtitle",
    "content": [
      "**Important** supporting point",
      "Another concise point",
      "**Key** takeaway"
    ],
    "slide_type": "title|content|cards|comparison|timeline|image|conclusion",
    "image_query": "2-5 word stock photo search query"
  }}
]

RULES:

1. Slide 1 MUST be title.
2. Final slide MUST be conclusion.
3. Every other slide MUST have an image_query.
4. Use varied layouts.
5. Do not make every slide a simple bullet list.
6. Use cards for categories.
7. Use comparison for differences.
8. Use timeline for steps/process.
9. Use image for a major visual concept.
10. Use content for explanation.
11. Keep 3-4 points per slide.
12. Keep bullets concise.
13. Keep bullets below 14 words.
14. Wrap 1-2 important words in **double asterisks**.
15. image_query must describe a concrete visual subject.
16. Avoid vague terms like:
    success, strategy, future, growth, intelligence.
17. Prefer:
    documents, people, offices, maps, passports,
    government buildings, charts, business meetings,
    finance documents, travel, computers.
18. Do not use emojis.
19. Do not add Markdown outside the JSON.
20. Return raw JSON only.
"""

        try:

            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config={
                    "response_mime_type": "application/json"
                }
            )

            text = response.text.strip()

            outline = json.loads(text)

            if not isinstance(outline, list):
                raise ValueError(
                    "Gemini returned an invalid outline"
                )

            return outline

        except Exception as e:

            print(
                f"❌ Gemini error: {e}"
            )

            return None

    # ========================================================
    # DOWNLOAD IMAGE FROM PEXELS
    # ========================================================
    def download_image(
        self,
        query,
        save_path="temp_image.jpg"
    ):

        try:

            print(
                f"   -> Searching Pexels for: '{query}'"
            )

            headers = {
                "Authorization": self.pexels_api_key
            }

            params = {
                "query": query,
                "per_page": 5,
                "orientation": "landscape"
            }

            # IMPORTANT:
            # This MUST be a normal URL.
            url = "https://api.pexels.com/v1/search"

            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=20
            )

            if response.status_code == 401:

                print(
                    "   ❌ Invalid Pexels API key."
                )

                return None

            response.raise_for_status()

            data = response.json()

            photos = data.get(
                "photos",
                []
            )

            # --------------------------------------------
            # Fallback
            # --------------------------------------------
            if not photos:

                print(
                    "   ⚠️ No images found. "
                    "Trying fallback..."
                )

                fallback_queries = [
                    "business documents",
                    "professional office",
                    "business meeting"
                ]

                for fallback in fallback_queries:

                    params["query"] = fallback

                    response = requests.get(
                        url,
                        params=params,
                        headers=headers,
                        timeout=20
                    )

                    if response.status_code != 200:
                        continue

                    data = response.json()

                    photos = data.get(
                        "photos",
                        []
                    )

                    if photos:
                        break

            if not photos:

                print(
                    "   ❌ No Pexels images found."
                )

                return None

            # --------------------------------------------
            # Image URL
            # --------------------------------------------
            image_url = photos[0]["src"]["large"]

            image_response = requests.get(
                image_url,
                timeout=20
            )

            image_response.raise_for_status()

            # --------------------------------------------
            # Save
            # --------------------------------------------
            with open(
                save_path,
                "wb"
            ) as file:

                file.write(
                    image_response.content
                )

            if not os.path.exists(save_path):

                print(
                    "   ❌ Image file was not created."
                )

                return None

            file_size = os.path.getsize(
                save_path
            )

            if file_size < 1000:

                print(
                    "   ❌ Downloaded image is invalid."
                )

                return None

            print(
                f"   ✅ Image downloaded "
                f"({file_size:,} bytes)"
            )

            return save_path

        except Exception as e:

            print(
                f"   ❌ Pexels error: {e}"
            )

            return None

    # ========================================================
    # TITLE SLIDE
    # ========================================================
    def create_title_slide(
        self,
        title,
        subtitle,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(
            slide,
            self.colors["dark"]
        )

        # Decorative blocks
        shape = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE,
            Inches(0),
            Inches(0),
            Inches(0.12),
            Inches(7.5)
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = self.colors["accent"]
        shape.line.fill.background()

        # Label
        self._add_text(
            slide,
            "ACADEMIC PRESENTATION",
            Inches(0.85),
            Inches(1.1),
            Inches(4.5),
            Inches(0.4),
            font_size=11,
            bold=True,
            color=self.colors["accent"]
        )

        # Title
        self._add_text(
            slide,
            title,
            Inches(0.85),
            Inches(1.9),
            Inches(10.8),
            Inches(1.8),
            font_size=42,
            bold=True,
            color=self.colors["white"],
            font_name=self.title_font
        )

        # Accent
        self._add_accent(
            slide,
            Inches(0.85),
            Inches(4.1),
            Inches(1.2),
            Inches(0.08)
        )

        # Subtitle
        self._add_text(
            slide,
            subtitle,
            Inches(0.85),
            Inches(4.55),
            Inches(8.5),
            Inches(0.8),
            font_size=19,
            color=RGBColor(203, 213, 225)
        )

        # Decorative circle
        circle = slide.shapes.add_shape(
            MSO_SHAPE.OVAL,
            Inches(10.7),
            Inches(1.4),
            Inches(1.5),
            Inches(1.5)
        )

        circle.fill.solid()
        circle.fill.fore_color.rgb = self.colors["accent"]
        circle.fill.transparency = 15
        circle.line.fill.background()

        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # CONTENT SLIDE WITH IMAGE
    # ========================================================
    def create_content_slide(
        self,
        title,
        content,
        image_path=None,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(slide)

        # Header
        self._add_text(
            slide,
            title,
            Inches(0.7),
            Inches(0.4),
            Inches(10.6),
            Inches(0.7),
            font_size=30,
            bold=True,
            color=self.colors["primary"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.7),
            Inches(1.2),
            Inches(0.9),
            Inches(0.06)
        )

        # ----------------------------------------------------
        # IMAGE EXISTS
        # ----------------------------------------------------
        if image_path and os.path.exists(image_path):

            # Text card
            self._add_card(
                slide,
                Inches(0.7),
                Inches(1.65),
                Inches(5.25),
                Inches(4.9)
            )

            text_box = slide.shapes.add_textbox(
                Inches(1.05),
                Inches(2.0),
                Inches(4.55),
                Inches(4.2)
            )

            tf = text_box.text_frame
            tf.clear()
            tf.word_wrap = True

            for index, point in enumerate(
                content[:5]
            ):

                p = (
                    tf.paragraphs[0]
                    if index == 0
                    else tf.add_paragraph()
                )

                p.space_after = Pt(18)

                self._add_formatted_text(
                    p,
                    f"• {point}",
                    base_size=17
                )

            # Image card
            self._add_card(
                slide,
                Inches(6.25),
                Inches(1.65),
                Inches(6.35),
                Inches(4.9),
                fill_color=self.colors["dark"]
            )

            try:

                slide.shapes.add_picture(
                    image_path,
                    Inches(6.35),
                    Inches(1.75),
                    width=Inches(6.15),
                    height=Inches(4.7)
                )

            except Exception as e:

                print(
                    f"   ⚠️ Image insertion error: {e}"
                )

        # ----------------------------------------------------
        # NO IMAGE
        # ----------------------------------------------------
        else:

            self._add_card(
                slide,
                Inches(0.7),
                Inches(1.65),
                Inches(11.9),
                Inches(4.9)
            )

            text_box = slide.shapes.add_textbox(
                Inches(1.1),
                Inches(2.0),
                Inches(11.1),
                Inches(4.1)
            )

            tf = text_box.text_frame
            tf.clear()
            tf.word_wrap = True

            for index, point in enumerate(
                content[:6]
            ):

                p = (
                    tf.paragraphs[0]
                    if index == 0
                    else tf.add_paragraph()
                )

                p.space_after = Pt(19)

                self._add_formatted_text(
                    p,
                    f"• {point}",
                    base_size=19
                )

        self._add_footer(slide)
        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # CARDS SLIDE WITH IMAGE
    # ========================================================
    def create_cards_slide(
        self,
        title,
        content,
        image_path=None,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(slide)

        self._add_text(
            slide,
            title,
            Inches(0.7),
            Inches(0.4),
            Inches(10.5),
            Inches(0.7),
            font_size=30,
            bold=True,
            color=self.colors["primary"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.7),
            Inches(1.2),
            Inches(0.9),
            Inches(0.06)
        )

        # ----------------------------------------------------
        # IMAGE ON RIGHT
        # ----------------------------------------------------
        if image_path and os.path.exists(image_path):

            card_width = 2.75

            for i, point in enumerate(
                content[:3]
            ):

                left = (
                    0.7 +
                    i * 2.95
                )

                self._add_card(
                    slide,
                    Inches(left),
                    Inches(1.75),
                    Inches(card_width),
                    Inches(3.6)
                )

                self._add_text(
                    slide,
                    f"{i + 1:02d}",
                    Inches(left + 0.25),
                    Inches(2.0),
                    Inches(0.7),
                    Inches(0.4),
                    font_size=11,
                    bold=True,
                    color=self.colors["accent"]
                )

                text_box = slide.shapes.add_textbox(
                    Inches(left + 0.25),
                    Inches(2.7),
                    Inches(2.25),
                    Inches(2.0)
                )

                tf = text_box.text_frame
                tf.clear()
                tf.word_wrap = True

                p = tf.paragraphs[0]

                self._add_formatted_text(
                    p,
                    point,
                    base_size=15
                )

            # Image
            self._add_card(
                slide,
                Inches(9.65),
                Inches(1.75),
                Inches(2.95),
                Inches(3.6),
                fill_color=self.colors["dark"]
            )

            try:

                slide.shapes.add_picture(
                    image_path,
                    Inches(9.75),
                    Inches(1.85),
                    width=Inches(2.75),
                    height=Inches(3.4)
                )

            except Exception as e:

                print(
                    f"   ⚠️ Image insertion error: {e}"
                )

        # ----------------------------------------------------
        # NO IMAGE
        # ----------------------------------------------------
        else:

            count = min(
                len(content),
                4
            )

            if count == 0:
                count = 1

            gap = 0.25
            total_width = 11.9

            card_width = (
                total_width -
                gap * (count - 1)
            ) / count

            for i in range(count):

                left = (
                    0.7 +
                    i * (
                        card_width +
                        gap
                    )
                )

                self._add_card(
                    slide,
                    Inches(left),
                    Inches(1.8),
                    Inches(card_width),
                    Inches(4.7)
                )

                self._add_text(
                    slide,
                    f"{i + 1:02d}",
                    Inches(left + 0.3),
                    Inches(2.15),
                    Inches(0.7),
                    Inches(0.4),
                    font_size=11,
                    bold=True,
                    color=self.colors["accent"]
                )

                text_box = slide.shapes.add_textbox(
                    Inches(left + 0.3),
                    Inches(2.8),
                    Inches(card_width - 0.6),
                    Inches(2.5)
                )

                tf = text_box.text_frame
                tf.clear()
                tf.word_wrap = True

                p = tf.paragraphs[0]

                self._add_formatted_text(
                    p,
                    content[i],
                    base_size=16
                )

        self._add_footer(slide)
        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # COMPARISON SLIDE WITH IMAGE
    # ========================================================
    def create_comparison_slide(
        self,
        title,
        content,
        image_path=None,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(slide)

        self._add_text(
            slide,
            title,
            Inches(0.7),
            Inches(0.4),
            Inches(10.5),
            Inches(0.7),
            font_size=30,
            bold=True,
            color=self.colors["primary"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.7),
            Inches(1.2),
            Inches(0.9),
            Inches(0.06)
        )

        # Split content
        midpoint = max(
            1,
            len(content) // 2
        )

        left_content = content[:midpoint]
        right_content = content[midpoint:]

        # Left card
        self._add_card(
            slide,
            Inches(0.7),
            Inches(1.7),
            Inches(5.45),
            Inches(4.8)
        )

        # Right card
        self._add_card(
            slide,
            Inches(6.4),
            Inches(1.7),
            Inches(5.45),
            Inches(4.8)
        )

        self._add_text(
            slide,
            "CATEGORY A",
            Inches(1.05),
            Inches(2.05),
            Inches(3),
            Inches(0.4),
            font_size=11,
            bold=True,
            color=self.colors["accent"]
        )

        self._add_text(
            slide,
            "CATEGORY B",
            Inches(6.75),
            Inches(2.05),
            Inches(3),
            Inches(0.4),
            font_size=11,
            bold=True,
            color=self.colors["accent"]
        )

        # Left content
        left_box = slide.shapes.add_textbox(
            Inches(1.05),
            Inches(2.65),
            Inches(4.7),
            Inches(3.3)
        )

        tf = left_box.text_frame
        tf.clear()
        tf.word_wrap = True

        for i, point in enumerate(left_content):

            p = (
                tf.paragraphs[0]
                if i == 0
                else tf.add_paragraph()
            )

            p.space_after = Pt(16)

            self._add_formatted_text(
                p,
                f"• {point}",
                base_size=16
            )

        # Right content
        right_box = slide.shapes.add_textbox(
            Inches(6.75),
            Inches(2.65),
            Inches(4.7),
            Inches(3.3)
        )

        tf = right_box.text_frame
        tf.clear()
        tf.word_wrap = True

        for i, point in enumerate(right_content):

            p = (
                tf.paragraphs[0]
                if i == 0
                else tf.add_paragraph()
            )

            p.space_after = Pt(16)

            self._add_formatted_text(
                p,
                f"• {point}",
                base_size=16
            )

        self._add_footer(slide)
        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # TIMELINE
    # ========================================================
    def create_timeline_slide(
        self,
        title,
        content,
        image_path=None,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(slide)

        self._add_text(
            slide,
            title,
            Inches(0.7),
            Inches(0.4),
            Inches(10.5),
            Inches(0.7),
            font_size=30,
            bold=True,
            color=self.colors["primary"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.7),
            Inches(1.2),
            Inches(0.9),
            Inches(0.06)
        )

        # Timeline line
        line = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE,
            Inches(1.0),
            Inches(3.45),
            Inches(10.9),
            Inches(0.04)
        )

        line.fill.solid()
        line.fill.fore_color.rgb = self.colors["accent"]
        line.line.fill.background()

        count = min(
            len(content),
            4
        )

        if count == 0:
            count = 1

        positions = [
            1.25,
            4.1,
            6.95,
            9.8
        ]

        for i in range(count):

            x = positions[i]

            # Circle
            circle = slide.shapes.add_shape(
                MSO_SHAPE.OVAL,
                Inches(x),
                Inches(3.25),
                Inches(0.45),
                Inches(0.45)
            )

            circle.fill.solid()
            circle.fill.fore_color.rgb = self.colors["accent"]
            circle.line.fill.background()

            # Number
            self._add_text(
                slide,
                str(i + 1),
                Inches(x - 0.28),
                Inches(2.55),
                Inches(1),
                Inches(0.4),
                font_size=12,
                bold=True,
                color=self.colors["accent"],
                alignment=PP_ALIGN.CENTER
            )

            # Text
            box = slide.shapes.add_textbox(
                Inches(x - 0.65),
                Inches(4.05),
                Inches(2.0),
                Inches(1.6)
            )

            tf = box.text_frame
            tf.clear()
            tf.word_wrap = True

            p = tf.paragraphs[0]

            self._add_formatted_text(
                p,
                content[i],
                base_size=14
            )

            p.alignment = PP_ALIGN.CENTER

        # Optional image panel
        if image_path and os.path.exists(image_path):

            self._add_card(
                slide,
                Inches(10.7),
                Inches(1.45),
                Inches(1.9),
                Inches(1.35),
                fill_color=self.colors["dark"]
            )

            try:

                slide.shapes.add_picture(
                    image_path,
                    Inches(10.78),
                    Inches(1.53),
                    width=Inches(1.74),
                    height=Inches(1.19)
                )

            except Exception as e:

                print(
                    f"   ⚠️ Timeline image error: {e}"
                )

        self._add_footer(slide)
        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # FULL IMAGE SLIDE
    # ========================================================
    def create_image_slide(
        self,
        title,
        content,
        image_path,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(
            slide,
            self.colors["dark"]
        )

        # Image
        if image_path and os.path.exists(image_path):

            try:

                slide.shapes.add_picture(
                    image_path,
                    Inches(0),
                    Inches(0),
                    width=Inches(13.333),
                    height=Inches(7.5)
                )

            except Exception as e:

                print(
                    f"   ⚠️ Image placement error: {e}"
                )

        # Overlay
        overlay = slide.shapes.add_shape(
            MSO_SHAPE.RECTANGLE,
            Inches(0),
            Inches(0),
            Inches(13.333),
            Inches(7.5)
        )

        overlay.fill.solid()
        overlay.fill.fore_color.rgb = self.colors["dark"]
        overlay.fill.transparency = 32
        overlay.line.fill.background()

        # Title
        self._add_text(
            slide,
            title,
            Inches(0.9),
            Inches(1.0),
            Inches(10.8),
            Inches(1.4),
            font_size=40,
            bold=True,
            color=self.colors["white"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.9),
            Inches(2.65),
            Inches(1.2),
            Inches(0.08)
        )

        # Content card
        self._add_card(
            slide,
            Inches(0.9),
            Inches(3.1),
            Inches(6.3),
            Inches(2.25),
            fill_color=self.colors["dark_2"],
            border_color=RGBColor(71, 85, 105)
        )

        box = slide.shapes.add_textbox(
            Inches(1.25),
            Inches(3.45),
            Inches(5.6),
            Inches(1.6)
        )

        tf = box.text_frame
        tf.clear()
        tf.word_wrap = True

        for i, point in enumerate(
            content[:3]
        ):

            p = (
                tf.paragraphs[0]
                if i == 0
                else tf.add_paragraph()
            )

            p.space_after = Pt(10)

            self._add_formatted_text(
                p,
                f"• {point}",
                base_size=15,
                dark=True
            )

        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # CONCLUSION
    # ========================================================
    def create_conclusion_slide(
        self,
        title,
        content,
        slide_number=1
    ):

        slide = self.presentation.slides.add_slide(
            self.presentation.slide_layouts[6]
        )

        self._set_background(
            slide,
            self.colors["dark"]
        )

        self._add_text(
            slide,
            "KEY TAKEAWAY",
            Inches(0.9),
            Inches(1.0),
            Inches(4),
            Inches(0.4),
            font_size=12,
            bold=True,
            color=self.colors["accent"]
        )

        self._add_text(
            slide,
            title,
            Inches(0.9),
            Inches(1.7),
            Inches(10.5),
            Inches(1.5),
            font_size=40,
            bold=True,
            color=self.colors["white"],
            font_name=self.title_font
        )

        self._add_accent(
            slide,
            Inches(0.9),
            Inches(3.45),
            Inches(1.2),
            Inches(0.08)
        )

        self._add_card(
            slide,
            Inches(0.9),
            Inches(4.0),
            Inches(8.8),
            Inches(2.0),
            fill_color=self.colors["dark_2"],
            border_color=RGBColor(51, 65, 85)
        )

        box = slide.shapes.add_textbox(
            Inches(1.25),
            Inches(4.35),
            Inches(8.0),
            Inches(1.3)
        )

        tf = box.text_frame
        tf.clear()
        tf.word_wrap = True

        for i, point in enumerate(
            content[:3]
        ):

            p = (
                tf.paragraphs[0]
                if i == 0
                else tf.add_paragraph()
            )

            p.space_after = Pt(8)

            self._add_formatted_text(
                p,
                f"• {point}",
                base_size=15,
                dark=True
            )

        self._add_slide_number(
            slide,
            slide_number
        )

    # ========================================================
    # GENERATE PRESENTATION
    # ========================================================
    def generate_presentation(
        self,
        topic,
        num_slides=5,
        output_file="professional_presentation.pptx"
    ):

        print(
            f"Generating professional presentation: {topic}"
        )

        # ----------------------------------------------------
        # RESET PRESENTATION
        # ----------------------------------------------------
        self.presentation = Presentation()

        self.presentation.slide_width = Inches(13.333)
        self.presentation.slide_height = Inches(7.5)

        # ----------------------------------------------------
        # GENERATE OUTLINE
        # ----------------------------------------------------
        outline = self.generate_content_outline(
            topic,
            num_slides
        )

        if not outline:

            print(
                "❌ Failed to generate outline."
            )

            return None

        outline = outline[:num_slides]

        # ----------------------------------------------------
        # CREATE SLIDES
        # ----------------------------------------------------
        for index, slide_data in enumerate(
            outline
        ):

            slide_number = index + 1

            title = slide_data.get(
                "title",
                f"Slide {slide_number}"
            )

            subtitle = slide_data.get(
                "subtitle",
                topic
            )

            content = slide_data.get(
                "content",
                []
            )

            slide_type = slide_data.get(
                "slide_type",
                "content"
            )

            slide_type = str(
                slide_type
            ).lower().strip()

            image_query = slide_data.get(
                "image_query",
                ""
            )

            print()
            print(
                f"Creating slide {slide_number}: {title}"
            )
            print(
                f"   -> Layout: {slide_type}"
            )

            image_path = None

            # =================================================
            # TITLE
            # =================================================
            if (
                slide_number == 1
                or slide_type == "title"
            ):

                self.create_title_slide(
                    title,
                    subtitle,
                    slide_number
                )

            # =================================================
            # CONCLUSION
            # =================================================
            elif slide_type == "conclusion":

                self.create_conclusion_slide(
                    title,
                    content,
                    slide_number
                )

            # =================================================
            # ALL VISUAL SLIDES
            # =================================================
            else:

                # ------------------------------------------------
                # Guaranteed fallback
                # ------------------------------------------------
                if not image_query:

                    fallback_queries = [
                        "Indian tax documents",
                        "business documents office",
                        "government office India",
                        "professional business meeting",
                        "financial documents"
                    ]

                    image_query = fallback_queries[
                        index % len(
                            fallback_queries
                        )
                    ]

                print(
                    f"   -> Image query: '{image_query}'"
                )

                # ------------------------------------------------
                # Download image
                # ------------------------------------------------
                temp_filename = (
                    f"temp_slide_{slide_number}.jpg"
                )

                image_path = self.download_image(
                    image_query,
                    temp_filename
                )

                # ------------------------------------------------
                # IMAGE
                # ------------------------------------------------
                if slide_type == "image":

                    self.create_image_slide(
                        title=title,
                        content=content,
                        image_path=image_path,
                        slide_number=slide_number
                    )

                # ------------------------------------------------
                # CARDS
                # ------------------------------------------------
                elif slide_type == "cards":

                    self.create_cards_slide(
                        title=title,
                        content=content,
                        image_path=image_path,
                        slide_number=slide_number
                    )

                # ------------------------------------------------
                # COMPARISON
                # ------------------------------------------------
                elif slide_type == "comparison":

                    self.create_comparison_slide(
                        title=title,
                        content=content,
                        image_path=image_path,
                        slide_number=slide_number
                    )

                # ------------------------------------------------
                # TIMELINE
                # ------------------------------------------------
                elif slide_type == "timeline":

                    self.create_timeline_slide(
                        title=title,
                        content=content,
                        image_path=image_path,
                        slide_number=slide_number
                    )

                # ------------------------------------------------
                # NORMAL CONTENT
                # ------------------------------------------------
                else:

                    self.create_content_slide(
                        title=title,
                        content=content,
                        image_path=image_path,
                        slide_number=slide_number
                    )

            # ----------------------------------------------------
            # DELETE TEMP IMAGE
            # ----------------------------------------------------
            if (
                image_path
                and os.path.exists(image_path)
            ):

                try:

                    os.remove(
                        image_path
                    )

                    print(
                        f"   -> Temporary image removed"
                    )

                except Exception as e:

                    print(
                        f"   ⚠️ Could not remove temp image: {e}"
                    )

        # ========================================================
        # SAVE
        # ========================================================
        try:

            self.presentation.save(
                output_file
            )

            print()
            print(
                "=========================================="
            )
            print(
                "✅ PROFESSIONAL PRESENTATION CREATED"
            )
            print(
                f"📁 File: {output_file}"
            )
            print(
                "=========================================="
            )

            return output_file

        except Exception as e:

            print(
                f"❌ Failed to save PowerPoint: {e}"
            )

            return None



In [ ]:
pixel_api_key = ''
gemini_api_key =''

In [ ]:
if __name__ == "__main__":

    # Your existing API key variables
    GEMINI_API_KEY = gemini_api_key
    PEXELS_API_KEY = pixel_api_key

    # Create generator
    generator = ProfessionalPPTGenerator(
        api_key=GEMINI_API_KEY,
        pexels_api_key=PEXELS_API_KEY
    )

    # Generate presentation
    output = generator.generate_presentation(
        topic="Residential Status under Income Tax",
        num_slides=5,
        output_file="residential_status_professional.pptx"
    )

    print(
        "Output:",
        output
    )

Generating professional presentation: Residential Status under Income Tax

Creating slide 1: Residential Status under Income Tax
   -> Layout: title

Creating slide 2: Determining Residency Status
   -> Layout: cards
   -> Image query: 'people checking documents'
   -> Searching Pexels for: 'people checking documents'
   ✅ Image downloaded (64,929 bytes)
   -> Temporary image removed

Creating slide 3: Basic Residency Tests
   -> Layout: comparison
   -> Image query: 'calendar and passport'
   -> Searching Pexels for: 'calendar and passport'
   ✅ Image downloaded (59,278 bytes)
   -> Temporary image removed

Creating slide 4: Evaluation Workflow
   -> Layout: timeline
   -> Image query: 'finance documents on desk'
   -> Searching Pexels for: 'finance documents on desk'
   ✅ Image downloaded (95,483 bytes)
   -> Temporary image removed

Creating slide 5: Summary and Conclusion
   -> Layout: conclusion

✅ PROFESSIONAL PRESENTATION CREATED
📁 File: residential_status_professional.pptx
Outp